## Loading MD input data ##

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# import resicon
# from resicon import *

# import geostas
# import mdtraj as md
# import MDAnalysis as mda 
# from redpandda import *

In [341]:
def preprocess_protein_trajectory(prot_info, k_cluster=None):
  frames_count = prot_info[3]
  traj_array, k_cluster = preprocessing(prot_info,frames_count,k_cluster)
  return traj_array, k_cluster

load molecular dynamcis dataset:

must consist of the following files:
* xtc or dcd trajectory file
* pdb peptide file
* folder where first two files are stored
* frame count (optional, otherwise None)
* k clusters (optional, otherwise None)

In [ ]:
# md_trajectory_info = ['trajectory-3.xtc','fs-peptide.pdb','McGibbon/',None,None]
md_trajectory_info = ['1hhp.dcd','1hhp.pdb','HIV1Protease/',None,None]

#how many frames to process
frames_count = md_trajectory_info[3]

trajectory_file = md_trajectory_info[0].split()[0]
pdb_file = md_trajectory_info[1].split()[0]

# call MD-related preprocessing
traj_array, k_cluster = preprocess_protein_trajectory(md_trajectory_info)
np.savez("data/traj_data_fspeptide3.npz", traj_array=traj_array, k_cluster=k_cluster)

load non-MD datasets

In [2]:
def prepare_data_from_df(x, use_mean_preprocessing=True, group_by_obj_id=False):
    import pandas as pd
    import numpy as np  

    if use_mean_preprocessing:
        # Aggregate duplicates by taking the mean of coordinates
        x = x.groupby(['t', 'obj_id'], as_index=False).mean()

    tpoints = []
    df_points = []
    traj_array = []
    point_array = []

    # Group by time ('t') after ensuring data is sorted by 'obj_id'

    if group_by_obj_id:
        group_variable = "obj_id"
        sort_variable = "t"
    else:
        group_variable = "t"
        sort_variable = "obj_id"

    for g in x.sort_values([sort_variable], ascending=True).groupby(group_variable):
        tpoints.append(g[1].values)
        df_points.append(pd.DataFrame(g[1]))
        new_df = pd.DataFrame(g[1])

        # Extract trajectory data (x, y, z) and object IDs
        traj_array.append(np.array(new_df[['x', 'y', 'z']].values))
        point_array.append(new_df[['obj_id']].values)

    # Calculate the number of frames and objects
    frames_count = len(df_points[0]) if df_points else 0
    n_objects = len(df_points)

    return traj_array, point_array, frames_count, n_objects


In [3]:
substitutions = {'t':'frame', 'obj_id':'id','label':'cid','x':'x','y':'y'}

def format_cluster_df(df, substitutions, add_z=True):
    filtered_df = df[list(substitutions.values())]
    filtered_df = filtered_df.rename(columns={v: k for k, v in substitutions.items()})

    if add_z:
        if 'z' not in df.columns:
            filtered_df['z'] = 0

    return filtered_df

In [79]:
k_cluster = None
filename = "calovi_1800_3"
orig_df = pd.read_csv("/Users/work/Library/Mobile Documents/com~apple~CloudDocs/Desktop/ADesktop/Studium/PhD/DataMining/2025-COMET/COMET/full_data/"+filename+".csv")

In [80]:
def test(x, use_mean_preprocessing=True, group_by_obj_id=False):
    import pandas as pd
    import numpy as np  

    if use_mean_preprocessing:
        # Aggregate duplicates by taking the mean of coordinates
        x = x.groupby(['t', 'obj_id'], as_index=False).mean()

    tpoints = []
    df_points = []
    traj_array = []
    point_array = []

    # Group by time ('t') after ensuring data is sorted by 'obj_id'

    if group_by_obj_id:
        group_variable = "obj_id"
        sort_variable = "t"
    else:
        group_variable = "t"
        sort_variable = "obj_id"

    for g in x.sort_values([sort_variable], ascending=True).groupby(group_variable):
        tpoints.append(g[1].values)
        df_points.append(pd.DataFrame(g[1]))
        new_df = pd.DataFrame(g[1])

        # Extract trajectory data (x, y, z) and object IDs
        traj_array.append(np.array(new_df[['x', 'y', 'z']].values))
        point_array.append(new_df[['obj_id']].values)

    # Calculate the number of frames and objects
    frames_count = len(df_points[0]) if df_points else 0
    n_objects = len(df_points)

    return new_df

In [82]:
df = format_cluster_df(orig_df, substitutions)
test_df = test(df, group_by_obj_id=False)

In [71]:
test_df.shape

(34, 6)

In [76]:
import redpandda_general
df = format_cluster_df(orig_df, substitutions)
traj_array, point_array, frames_count, n_objects = redpandda_general.prepare_data_from_df(df)

In [57]:
print("n animals",n_objects)
print("n frames",frames_count)

n animals 3000
n frames 34


In [117]:
df = orig_df
df = df.drop_duplicates(subset=["id", "frame"], keep=False)
print("no timepoints",df["frame"].nunique())
print("no animals",df["id"].nunique())

no timepoints 1800
no animals 75


In [118]:
import numpy as np

# Ensure trajectories are sorted
df = df.sort_values(["id", "frame"])

# Extract unique animals
animal_ids = df["id"].unique()

traj_list = []
for aid in animal_ids:
    g = df[df["id"] == aid].sort_values("frame")
    traj = g[["x", "y"]].values           # shape (T, 2)
    
    traj_list.append(traj)
    if len(traj) != df["frame"].nunique():
        print("TEST",len(traj))
        a = traj
        break


# Convert to tslearn format
data_ts = np.stack(traj_list, axis=0)

print("Final shape:", data_ts.shape)
# → (n_animals, n_timepoints, 2)


Final shape: (75, 1800, 2)


In [20]:
true_labels = df.groupby("obj_id")["label"].first().to_numpy()
n_clusters = df["label"].nunique()



In [316]:
traj_array, point_array, frames_count, n_objects = prepare_data_from_df(df)

In [317]:
print("Number of timepoints",frames_count)
print("Number of animals",n_objects)

Number of timepoints 49
Number of animals 100


In [42]:
n_objects

100

In [294]:
np.savez("data/"+filename+"_traj_data.npz", traj_array=traj_array,frames_count=frames_count, n_objects=n_objects ) #, k_cluster=k_cluster)